## Importing Dependencies

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score, multilabel_confusion_matrix
import seaborn as sn
import matplotlib.pyplot as plt
import numpy as np

from collections import Counter

# The following five lines ensure that we reload the preprocessing functions 
# everytime this cell is called
import importlib
import helper_files.preprocessing as preprocFuncts
import helper_files.util as util
import parameters as MyParams
importlib.reload(preprocFuncts)
importlib.reload(util)
importlib.reload(MyParams)

# MODEL PARAMETERS
K_NEIGHBORS = 20

# DERIVATION PARAMETERS
WINDOW_LEN = MyParams.WINDOW_LEN # measured in IQ samples
FFT_OVERLAP = MyParams.FFT_OVERLAP
NUM_FEATURES = MyParams.NUM_FEATURES

PATCH_WIDTH_TIME = MyParams.PATCH_WIDTH_TIME
PATCH_HEIGHT_FREQ = MyParams.PATCH_HEIGHT_FREQ
TIME_HOP = MyParams.TIME_HOP
FREQ_HOP = MyParams.FREQ_HOP

# DATA PARAMETERS
TRAINING_DATA_DIR = MyParams.training_data_dir
EVAL_DATA_DIR = MyParams.eval_data_dir

PREPROCESSING_TECHNIQUE = MyParams.PREPROCESSING_TECHNIQUE

NUM_TRAINING_FILES = MyParams.NUM_TRAINING_FILES # how many files in the saved numpy data for training
NUM_EVALUATION_FILES = MyParams.NUM_EVALUATION_FILES # how many files in the saved numpy data for evaluation
MAX_FILES = MyParams.MAX_FILES # if not using the saved numpy data, this is the max files to intake
USE_SAVED_DATA = MyParams.USE_SAVED_DATA # True = used the saved .npy file data instead of re-deriving the features again
SAVE_METRICS_TO_FILE = MyParams.SAVE_METRICS_TO_FILE
TRAINING_DATASET = MyParams.TRAINING_DATASET
EVAL_DATASET = MyParams.EVAL_DATASET

FEATURES_TO_USE = MyParams.FEATURES_TO_USE

## Importing Training Datasets

In [ ]:
importlib.reload(preprocFuncts)
importlib.reload(util)
importlib.reload(MyParams)

importlib.reload(preprocFuncts)
importlib.reload(util)
importlib.reload(MyParams)
print("Pulling from directory: ", TRAINING_DATA_DIR)


TRAINING_POSTFIXES = [
    "dji_mini_4k_10_MHz_comms_room",
    "dji_mini_4k_10mhz_chamber",
    "dji_mini_4k_20_MHz_comms_room",
    "dji_mini_4k_20mhz_chamber",
    "holystone_360s_RF_chamber_DL",
    "n11_pro_comms_room_DL",
    "n11_pro_comms_room_UL",
    "n11_pro_rf_chamber_DL",
    "n11_pro_rf_chamber_UL",
    "ruko_f11_mini_comms_room_DL",
    "ruko_f11_mini_comms_room_UL",
    "ruko_f11_mini_rf_chamber_DL",
    "ruko_f11_mini_rf_chamber_UL",
]

EVAL_POSTFIXES = [
    "dji_mini_4k_10_MHz_comms_room",
    "dji_mini_4k_10mhz_chamber",
    "dji_mini_4k_20_MHz_comms_room",
    "dji_mini_4k_20mhz_chamber",
    "holystone_360s_RF_chamber_DL",
    "n11_pro_comms_room",
    "n11_pro_rf_chamber",
    "ruko_f11_mini_comms_room_DL",
    "ruko_f11_mini_comms_room_UL",
    "ruko_f11_mini_rf_chamber_DL",
    "ruko_f11_mini_rf_chamber_UL",
]

TRAINING_FEATURES = {}
EVAL_FEATURES = {}

for d in TRAINING_POSTFIXES:
    print(d)
    TRAINING_FEATURES[d] = preprocFuncts.preprocess_files(
        patch_width_time=PATCH_WIDTH_TIME,
        patch_height_freq=PATCH_HEIGHT_FREQ,
        time_hop=TIME_HOP,
        freq_hop=FREQ_HOP,
        postfix=f"train_6ftrs_100files_1024win_010over_{d}",
        data_dir=TRAINING_DATA_DIR,
        features_to_use=FEATURES_TO_USE,
        window_len=WINDOW_LEN,
        overlap=FFT_OVERLAP,
        saved_data=True
    )

for d in EVAL_POSTFIXES:
    print(d)
    EVAL_FEATURES[d] = preprocFuncts.preprocess_files(
        patch_width_time=PATCH_WIDTH_TIME,
        patch_height_freq=PATCH_HEIGHT_FREQ,
        time_hop=TIME_HOP,
        freq_hop=FREQ_HOP,
        postfix=f"eval_6ftrs_100files_1024win_010over_{d}",
        data_dir=TRAINING_DATA_DIR,
        features_to_use=FEATURES_TO_USE,
        window_len=WINDOW_LEN,
        overlap=FFT_OVERLAP,
        saved_data=True
    )

training_samples = np.vstack(
    [TRAINING_FEATURES[d][0] for d in TRAINING_POSTFIXES] +
    [EVAL_FEATURES[d][0] for d in EVAL_POSTFIXES]
)
training_labels = np.concatenate(
    [TRAINING_FEATURES[d][1] for d in TRAINING_POSTFIXES] +
    [EVAL_FEATURES[d][1] for d in EVAL_POSTFIXES]
)


print()
print("Training sample shape:", np.shape(training_samples))
print("Traning label shape:", np.shape(training_labels))
print()

unique_labels, counts = np.unique(training_labels, return_counts=True)

print("The counts:")
for lab, c in zip(unique_labels, counts):
    print(f"{str(f'{c:,}'):8s} \"{lab}\"")

## Preprocessing

In [ ]:
print("\n==BEFORE BALANCING========")
util.display(training_samples, training_labels)

training_derived_samples, training_labels = preprocFuncts.balance_by_median(training_samples, training_labels, unlabeled_downsampling=7_200_000)

print("\n\n==AFTER BALANCING========")
util.display(training_derived_samples, training_labels)

X = training_derived_samples
y = training_labels

print(f"Number of samples: {X.shape[0]:,}")
print(f"Number of labels: {len(y):,}")

remove_labels = [
    'Bluetooth',
    'WIFI',
    'reflection'
]

# Build mask: True = keep, False = remove
train_mask = ~np.isin(y, remove_labels)
eval_mask = ~np.isin(y, remove_labels)

# Apply mask
X = X[train_mask]
y = y[train_mask]
y_strings = y


le = LabelEncoder()
y_train = le.fit_transform(y)

## Initializing Models

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from joblib import Parallel, delayed

from sklearn.base import clone
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    confusion_matrix, f1_score, accuracy_score, 
    roc_curve, auc, roc_auc_score
)


# Define Folds (Pre-generate to ensure consistency)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=67)
folds = list(skf.split(X, y))

# Model Dictionary
# Note: probability=True is required for SVM to generate ROC curves
models = {
    "GaussianNB": GaussianNB(),
    "KNN": KNeighborsClassifier(n_neighbors=K_NEIGHBORS),
    "RandomForest": RandomForestClassifier(n_estimators=200, random_state=67, class_weight='balanced'),
    "XGBoost": GradientBoostingClassifier(n_estimators=200, random_state=67),
    "MLP-Shallow": MLPClassifier(
        hidden_layer_sizes=(256),  # One hidden layer with 256 neurons
        activation='relu',
        solver='adam',
        max_iter=300,
        random_state=67,
        early_stopping=True,
        n_iter_no_change=10,
    ),
    "MLP-Deep": MLPClassifier(
        hidden_layer_sizes=(256, 128, 64),  # Three hidden layers with 128, 64, and 32 neurons
        activation='relu',
        solver='adam',
        max_iter=300,
        random_state=67,
        early_stopping=True,
        n_iter_no_change=10,
    ),
}

# 4. Global storage for comparison
results_summary = {} # Stores {model_name: {metric_name: value}}
roc_data = {}

## Model 1: Gaussian Naive Bayes

In [ ]:
from sklearn.preprocessing import LabelBinarizer

model_key = "GaussianNB"
current_model = models[model_key]

def evaluate_fold(train_idx, test_idx, model, X, y):
    X_tr, X_te = X[train_idx], X[test_idx]
    y_tr, y_te = y[train_idx], y[test_idx]
    
    m = clone(model)
    m.fit(X_tr, y_tr)
    
    y_pred = m.predict(X_te)
    # MULTICLASS CHANGE: Return probabilities for ALL classes
    y_prob = m.predict_proba(X_te) 
    
    return {
        "cm": confusion_matrix(y_te, y_pred),
        "f1": f1_score(y_te, y_pred, average='macro'),
        "acc": accuracy_score(y_te, y_pred),
        "y_te": y_te,
        "y_prob": y_prob
    }

# Parallel Execution
fold_results = Parallel(n_jobs=-1)(
    delayed(evaluate_fold)(tr, te, current_model, X, y) for tr, te in folds
)

# Aggregating Results
agg_cm = sum(res['cm'] for res in fold_results)
avg_f1 = np.mean([res['f1'] for res in fold_results])
avg_acc = np.mean([res['acc'] for res in fold_results])

# MULTICLASS ROC/AUC LOGIC
all_y_true = np.concatenate([res['y_te'] for res in fold_results])
all_y_probs = np.concatenate([res['y_prob'] for res in fold_results])

# 1. Calculate Multiclass AUC (Macro-averaged One-vs-Rest)
# 'ovr' = One-vs-Rest, 'macro' = average the scores of each class
multi_auc = roc_auc_score(all_y_true, all_y_probs, multi_class='ovr', average='macro')

# 2. For the ROC Curve Plot (Macro-average approach)
# We binarize the output to calculate a single representative curve
lb = LabelBinarizer()
lb.fit(all_y_true)
y_true_bin = lb.transform(all_y_true)
fpr, tpr, _ = roc_curve(y_true_bin.ravel(), all_y_probs.ravel())

# Save results
results_summary[model_key] = {"F1 Macro": avg_f1, "Accuracy": avg_acc, "AUC": multi_auc}
roc_data[model_key] = {"fpr": fpr, "tpr": tpr, "auc": multi_auc, "cm": agg_cm}

print(f"Finished {model_key}")
print("--------------------------------")
print(f"F1: {avg_f1:.4f} | Acc: {avg_acc:.4f} | AUC: {multi_auc:.4f}")

In [ ]:
plt.figure(figsize=(10, 8))
sn.heatmap(
    roc_data[model_key]["cm"],
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=np.unique(y_strings),
    yticklabels=np.unique(y_strings),
)

plt.title('5-Fold Aggregated Confusion Matrix (Gaussian Naive Bayes)')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.xticks(rotation=90)

if SAVE_METRICS_TO_FILE:
    plt.savefig(f"./metrics/iq_gnb_cm.png", bbox_inches='tight', dpi=200)
plt.show()

## Comparisons

### 1. Accuracy and F1 Macro Scores:

In [ ]:
df_results = pd.DataFrame(results_summary).T
ax = df_results[['Accuracy']].plot(kind='bar', figsize=(10, 6), legend=False)
plt.title("Model Comparison: Accuracy")
plt.ylabel("Score")
plt.xticks(rotation=0)
plt.ylim(0, 1.05)

plt.bar_label(ax.containers[0], fmt='%.3f', padding=5, fontsize=10)
plt.grid(axis='y', linestyle='--', alpha=0.3)

if SAVE_METRICS_TO_FILE:
    plt.savefig("./metrics/iq_overall_accuracy.png", bbox_inches='tight', dpi=200)

plt.show()

### 2. F1 Score (Macro)

In [ ]:
df_results = pd.DataFrame(results_summary).T
ax = df_results[['F1 Macro']].plot(kind='bar', figsize=(10, 6), legend=False)
plt.title("Model Comparison: F1 Score (Macro)")
plt.ylabel("Score")
plt.xticks(rotation=0)
plt.ylim(0, 1.05)

plt.bar_label(ax.containers[0], fmt='%.3f', padding=5, fontsize=10)
plt.grid(axis='y', linestyle='--', alpha=0.3)

if SAVE_METRICS_TO_FILE:
    plt.savefig("./metrics/iq_overall_f1.png", bbox_inches='tight', dpi=200)
plt.show()

### 2. ROC Curves

In [ ]:
plt.figure(figsize=(10, 7))

for model_name, data in roc_data.items():
    plt.plot(data['fpr'], data['tpr'], label=f"{model_name}")

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve Comparison')
plt.legend(loc="lower right")
plt.grid(alpha=0.3)

if SAVE_METRICS_TO_FILE:
    plt.savefig("./metrics/iq_overall_roc.png", bbox_inches='tight', dpi=200)
plt.show()

### 4. AUC Values

In [ ]:
df_results = pd.DataFrame(results_summary).T
ax = df_results[['AUC']].plot(kind='bar', figsize=(10, 6), legend=False)
plt.title("Model Comparison: AUC Values")
plt.ylabel("Score")
plt.xticks(rotation=0)
plt.ylim(0, 1.05)

plt.bar_label(ax.containers[0], fmt='%.3f', padding=5, fontsize=10)
plt.grid(axis='y', linestyle='--', alpha=0.3)

if SAVE_METRICS_TO_FILE:
    plt.savefig("./metrics/iq_overall_auc.png", bbox_inches='tight', dpi=200)
plt.show()